# Advanced Vision Ensemble for House Recognition
## การจำแนกบ้านขั้นสูงด้วยเอนเซมเบิล

This notebook implements an advanced vision ensemble model for house recognition using multiple pre-trained architectures with test-time augmentation (TTA).

**การจำแนก**: 
- EfficientNet-B3 ด้วย transfer learning
- Two-stage fine-tuning (head-only → full model)
- Test-time augmentation สำหรับความแม่นยำที่สูงขึ้น
- Weighted ensemble blending

## 1. ตั้งค่าสภาพแวดล้อม (Environment Setup)

นำเข้าไลบรารีที่จำเป็นและตั้งค่าการเชื่อมต่อ Kaggle

In [ ]:
# ติดตั้ง kaggle API
!pip install -q kaggle

In [ ]:
# ตั้งค่า Kaggle credentials
# โปรดอัพโหลดไฟล์ kaggle.json ของคุณก่อนรันเซลล์นี้
from google.colab import files
import os

print("โปรดอัพโหลดไฟล์ kaggle.json ของคุณ...")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# ดาวน์โหลดข้อมูลจาก Kaggle
import subprocess
from pathlib import Path

competition_slug = "super-ai-engineer-season-6-individual-hackathon-house-recognition"
!kaggle competitions download -c {competition_slug}

# แตกไฟล์
!unzip -q {competition_slug}.zip

DATA_DIR = Path("/content") / competition_slug
print(f"ข้อมูลใหม่ถูกดาวน์โหลดไปยัง: {DATA_DIR}")

## 2. นำเข้าไลบรารี (Import Libraries)

นำเข้าทั้งหมดที่จำเป็นสำหรับการฝึกอบรมและการทำนาย

In [ ]:
import json
import math
import random
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchvision.transforms as T
import matplotlib.pyplot as plt
from collections import Counter

print(f"PyTorch เวอร์ชัน: {torch.__version__}")
print(f"CUDA พร้อมใช้งาน: {torch.cuda.is_available()}")

## 3. การตั้งค่าพารามิเตอร์ (Configuration)

กำหนดค่าคงที่และพารามิเตอร์สำหรับการฝึกอบรม

In [ ]:
# ===== พารามิเตอร์ทั่วไป =====
SEED = 42
VAL_SIZE = 0.15
NUM_WORKERS = 2
TTA_PASSES = 4

# ตั้งค่า device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"ใช้ device: {device}")

# ตั้งค่า seed เพื่อความสอดคล้องกัน
def seed_everything(seed: int = SEED) -> None:
    """ตั้งค่า seed สำหรับ reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.backends.cudnn.benchmark = True

## 4. ฟังก์ชั่นสนับสนุน (Utility Functions)

ฟังก์ชั่นสำหรับการบันทึก, โหลดข้อมูล และจัดการเส้นทาง

In [ ]:
def log(msg: str) -> None:
    """พิมพ์ข้อความพร้อมเวลา"""
    now = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{now}] {msg}", flush=True)

log("เริ่มต้นชุดข้อมูล...")

In [ ]:
def build_image_paths(data_dir: Path):
    """สร้างเส้นทางรูปภาพและป้ายกำกับจากไดเรกทอรีข้อมูล"""
    train_csv = data_dir / "train.csv"
    sample_sub = data_dir / "sample_submission.csv"
    train_dir = data_dir / "train" / "train"
    test_dir = data_dir / "test" / "test"

    train_df = pd.read_csv(train_csv)
    sample_df = pd.read_csv(sample_sub)

    train_paths = [train_dir / name for name in train_df["image_name"].astype(str)]
    train_labels = train_df["class"].astype(int).to_numpy()
    test_ids = sample_df["id"].astype(str).tolist()
    test_paths = [test_dir / f"{img_id}.jpg" for img_id in test_ids]

    return train_df, sample_df, train_paths, train_labels, test_paths

# โหลดข้อมูล
train_df, sample_df, train_paths, train_labels, test_paths = build_image_paths(DATA_DIR)
log(f"โหลดรูปภาพ: {len(train_paths)} ภาพฝึก, {len(test_paths)} ภาพทดสอบ")

## 5. การวิเคราะห์ข้อมูลเบื้องต้น (EDA)

ตรวจสอบการกระจายของคลาส และแสดงตัวอย่างรูปภาพ

In [ ]:
# การกระจายของคลาส
class_counts = Counter(train_labels)
print(f"\nการกระจายของคลาส:")
for cls, count in sorted(class_counts.items()):
    pct = count / len(train_labels) * 100
    print(f"  คลาส {cls}: {count} ({pct:.2f}%)")

# วาดแผนภูมิ
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
classes = sorted(class_counts.keys())
counts = [class_counts[c] for c in classes]
ax.bar(classes, counts, color='steelblue', alpha=0.8)
ax.set_xlabel('คลาส (Class)')
ax.set_ylabel('จำนวน (Count)')
ax.set_title('การกระจายของคลาสในชุดข้อมูลฝึก')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# แสดงตัวอย่างรูปภาพ
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

# เลือกรูปภาพสุ่มจากแต่ละคลาส
sample_indices = []
for cls in sorted(class_counts.keys()):
    cls_indices = np.where(train_labels == cls)[0]
    sample_indices.extend(np.random.choice(cls_indices, size=min(3, len(cls_indices)), replace=False))

for idx, ax in enumerate(axes):
    if idx < len(sample_indices):
        img_path = train_paths[sample_indices[idx]]
        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
        ax.set_title(f'คลาส: {train_labels[sample_indices[idx]]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Dataset และ DataLoader (Dataset & Transforms)

นิยามคลาสชุดข้อมูลและฟังก์ชั่น transform

In [ ]:
class HouseDataset(Dataset):
    """ชุดข้อมูลของรูปภาพบ้าน"""
    def __init__(self, paths, labels=None, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        if self.labels is None:
            return img
        return img, torch.tensor(self.labels[idx], dtype=torch.long)

In [ ]:
def image_stats_from_weights(weight_enum):
    """ดึงสถิติการทำให้เป็นมาตรฐาน (mean/std) จากน้ำหนักที่บันทึกไว้"""
    mean = weight_enum.transforms().mean
    std = weight_enum.transforms().std
    return mean, std

def build_transforms(cfg, weights):
    """สร้าง transform pipeline สำหรับการฝึกอบรม, การตรวจสอบ และ TTA"""
    mean, std = image_stats_from_weights(weights)
    size = cfg["img_size"]
    
    # Transform สำหรับการฝึก
    train_tfm = T.Compose([
        T.RandomResizedCrop(size, scale=(0.8, 1.0), ratio=(0.9, 1.1), 
                           interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.22, contrast=0.22, saturation=0.18, hue=0.04),
        T.RandomRotation(8),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    
    # Transform สำหรับการตรวจสอบ
    val_tfm = T.Compose([
        T.Resize(int(size * 1.12), interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(size),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    
    # Transform สำหรับ Test-Time Augmentation
    tta_tfms = [
        val_tfm,
        T.Compose([
            T.Resize(int(size * 1.16), interpolation=T.InterpolationMode.BICUBIC),
            T.RandomCrop(size),
            T.RandomHorizontalFlip(p=1.0),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ]),
        T.Compose([
            T.Resize(int(size * 1.12), interpolation=T.InterpolationMode.BICUBIC),
            T.FiveCrop(size),
            T.Lambda(lambda crops: T.functional.to_tensor(crops[0])),
            T.Normalize(mean=mean, std=std),
        ]),
        T.Compose([
            T.Resize(int(size * 1.20), interpolation=T.InterpolationMode.BICUBIC),
            T.RandomResizedCrop(size, scale=(0.9, 1.0), ratio=(0.95, 1.05), 
                               interpolation=T.InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ]),
    ]
    
    return train_tfm, val_tfm, tta_tfms[:TTA_PASSES]

def make_loaders(train_paths, train_labels, val_paths, val_labels, 
                 train_tfm, val_tfm, batch_size):
    """สร้าง DataLoader สำหรับการฝึกอบรมและการตรวจสอบ"""
    train_ds = HouseDataset(train_paths, train_labels, transform=train_tfm)
    val_ds = HouseDataset(val_paths, val_labels, transform=val_tfm)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, 
                             num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, 
                           num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader

## 7. โครงสร้างโมเดล (Model Architecture)

โหลดโมเดลที่ฝึกอบรมแล้วและปรับแต่ง head

In [ ]:
def load_model(cfg):
    """โหลดโมเดลพร้อมน้ำหนักที่ฝึกอบรมแล้ว"""
    name = cfg["name"]
    
    if name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.35, inplace=True),
            nn.Linear(in_features, 1)
        )
        return model, weights

    if name == "efficientnet_b3":
        weights = models.EfficientNet_B3_Weights.DEFAULT
        model = models.efficientnet_b3(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.35, inplace=True),
            nn.Linear(in_features, 1)
        )
        return model, weights

    if name == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT
        model = models.convnext_tiny(weights=weights)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, 1)
        return model, weights

    raise ValueError(f"โมเดลที่ไม่รองรับ: {name}")

## 8. ฟังก์ชั่นการฝึกอบรม (Training Functions)

ฟังก์ชั่นหลักสำหรับการฝึกอบรมและการทำนาย

In [ ]:
def freeze_backbone(model):
    """หยุดการฝึกอบรมของ backbone และ unfreeze classifier"""
    for p in model.parameters():
        p.requires_grad = False
    if hasattr(model, "classifier"):
        for p in model.classifier.parameters():
            p.requires_grad = True

def unfreeze_all(model):
    """เปิดใหญ่การฝึกอบรมทั้งหมด"""
    for p in model.parameters():
        p.requires_grad = True

def train_one_epoch(model, loader, optimizer, scaler, criterion, device, 
                   log_prefix="train", log_every=40):
    """ฝึกอบรมโมเดลสำหรับ epoch เดียว"""
    model.train()
    losses = []
    for batch_idx, (images, labels) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        with autocast("cuda", enabled=device.type == "cuda"):
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
        if batch_idx % log_every == 0 or batch_idx == len(loader):
            avg_loss = np.mean(losses[-min(len(losses), 20):])
            log(f"{log_prefix} batch {batch_idx}/{len(loader)} loss={avg_loss:.4f}")
    return float(np.mean(losses))

In [ ]:
@torch.no_grad()
def predict_probs(model, loader, device):
    """ทำนายความน่าจะเป็นบนชุดข้อมูล"""
    model.eval()
    all_probs = []
    for batch in loader:
        if isinstance(batch, (list, tuple)):
            images = batch[0]
        else:
            images = batch
        images = images.to(device, non_blocking=True)
        with autocast("cuda", enabled=device.type == "cuda"):
            logits = model(images).squeeze(1)
        probs = torch.sigmoid(logits).float().cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)

def predict_with_tta(model, paths, tta_tfms, batch_size, device):
    """ทำนายโดยใช้ Test-Time Augmentation"""
    probs = []
    for tfm in tta_tfms:
        ds = HouseDataset(paths, labels=None, transform=tfm)
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False, 
                       num_workers=NUM_WORKERS, pin_memory=True)
        probs.append(predict_probs(model, dl, device))
    return np.mean(probs, axis=0)

def find_best_threshold(y_true, probs):
    """ค้นหา threshold ที่ดีที่สุดสำหรับความแม่นยำสูงสุด"""
    best_acc = 0.0
    best_thr = 0.5
    for thr in np.arange(0.3, 0.71, 0.01):
        acc = accuracy_score(y_true, (probs > thr).astype(int))
        if acc > best_acc:
            best_acc = float(acc)
            best_thr = float(thr)
    return best_acc, best_thr

## 9. ฟังก์ชั่นฝึกอบรมหลัก (Main Training Function)

ฟังก์ชั่นรวมที่จัดการสองขั้นตอนของการปรับแต่งและ TTA

In [ ]:
def train_model(cfg, train_paths, train_labels, val_paths, val_labels, device):
    """ฝึกอบรมโมเดลพร้อมสองขั้นตอน (head-only, then full)"""
    log(f"\n{'='*60}")
    log(f"ฝึกอบรมโมเดล: {cfg['name']}")
    log(f"{'='*60}")
    
    model, weights = load_model(cfg)
    train_tfm, val_tfm, tta_tfms = build_transforms(cfg, weights)
    train_loader, val_loader = make_loaders(train_paths, train_labels, 
                                            val_paths, val_labels, 
                                            train_tfm, val_tfm, cfg["batch_size"])
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    scaler = GradScaler("cuda", enabled=device.type == "cuda")

    best_state = None
    best_acc = -1.0
    best_thr = 0.5
    best_val_probs = None
    best_epoch = -1

    # ===== ขั้นตอนที่ 1: Fine-tuning เฉพาะ head =====
    log("\n[ขั้นตอนที่ 1] ฝึกอบรมเฉพาะ head...")
    freeze_backbone(model)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["head_lr"],
        weight_decay=1e-4
    )
    for epoch in range(cfg["head_epochs"]):
        loss = train_one_epoch(model, train_loader, optimizer, scaler, 
                              criterion, device, 
                              log_prefix=f"{cfg['name']} head[{epoch+1}]")
        val_probs = predict_probs(model, val_loader, device)
        val_acc, val_thr = find_best_threshold(val_labels, val_probs)
        log(f"{cfg['name']} head epoch {epoch+1}/{cfg['head_epochs']} "
            f"loss={loss:.4f} val_acc={val_acc:.4f} thr={val_thr:.2f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_thr = val_thr
            best_state = deepcopy(model.state_dict())
            best_val_probs = val_probs.copy()
            best_epoch = epoch + 1

    # ===== ขั้นตอนที่ 2: Fine-tuning แบบเต็ม =====
    log("\n[ขั้นตอนที่ 2] ฝึกอบรมแบบเต็ม...")
    unfreeze_all(model)
    optimizer = torch.optim.AdamW(
        [{"params": model.parameters(), "lr": cfg["full_lr"]}],
        weight_decay=cfg["weight_decay"]
    )
    total_steps = max(1, len(train_loader) * cfg["full_epochs"])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=cfg["full_lr"],
        total_steps=total_steps,
        pct_start=0.2,
        anneal_strategy="cos",
        div_factor=10.0,
        final_div_factor=50.0
    )

    for epoch in range(cfg["full_epochs"]):
        model.train()
        losses = []
        for batch_idx, (images, labels) in enumerate(train_loader, start=1):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float()
            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=device.type == "cuda"):
                logits = model(images).squeeze(1)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(loss.item())
            if batch_idx % 40 == 0 or batch_idx == len(train_loader):
                avg_loss = np.mean(losses[-min(len(losses), 20):])
                log(f"{cfg['name']} full[{epoch+1}] batch {batch_idx}/{len(train_loader)} "
                    f"loss={avg_loss:.4f}")

        val_probs = predict_probs(model, val_loader, device)
        val_acc, val_thr = find_best_threshold(val_labels, val_probs)
        mean_loss = float(np.mean(losses))
        log(f"{cfg['name']} full epoch {epoch+1}/{cfg['full_epochs']} "
            f"loss={mean_loss:.4f} val_acc={val_acc:.4f} thr={val_thr:.2f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_thr = val_thr
            best_state = deepcopy(model.state_dict())
            best_val_probs = val_probs.copy()
            best_epoch = cfg["head_epochs"] + epoch + 1

    # ===== TTA Evaluation =====
    log("\n[TTA] ประเมินด้วย Test-Time Augmentation...")
    model.load_state_dict(best_state)
    val_probs_tta = predict_with_tta(model, val_paths, tta_tfms, 
                                     cfg["batch_size"], device)
    tta_acc, tta_thr = find_best_threshold(val_labels, val_probs_tta)
    if tta_acc >= best_acc:
        log(f"TTA ปรับปรุง: {best_acc:.4f} → {tta_acc:.4f}")
        best_acc = tta_acc
        best_thr = tta_thr
        best_val_probs = val_probs_tta.copy()

    log(f"\nสรุป {cfg['name']}:")
    log(f"  Epoch ที่ดีที่สุด: {best_epoch}")
    log(f"  ความแม่นยำ: {best_acc:.4f}")
    log(f"  Threshold: {best_thr:.4f}")

    return {
        "name": cfg["name"],
        "model": model,
        "weights": weights,
        "tta_tfms": tta_tfms,
        "batch_size": cfg["batch_size"],
        "val_acc": best_acc,
        "val_thr": best_thr,
        "val_probs": best_val_probs,
        "best_epoch": best_epoch,
    }

## 10. เตรียมข้อมูล (Data Preparation)

แบ่งข้อมูลเป็นชุดฝึก, ตรวจสอบ และทดสอบ

In [ ]:
# แบ่งชุดข้อมูล
idx = np.arange(len(train_paths))
train_idx, val_idx = train_test_split(
    idx,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_labels
)

train_paths_split = [train_paths[i] for i in train_idx]
val_paths_split = [train_paths[i] for i in val_idx]
train_labels_split = train_labels[train_idx]
val_labels_split = train_labels[val_idx]

log(f"\nแบ่งข้อมูล:")
log(f"  ชุดฝึก: {len(train_paths_split)} ภาพ")
log(f"  ชุดตรวจสอบ: {len(val_paths_split)} ภาพ")
log(f"  ชุดทดสอบ: {len(test_paths)} ภาพ")

## 11. การฝึกอบรมโมเดล (Model Training)

ฝึกอบรมชุดโมเดล

In [ ]:
# กำหนดค่าการฝึกอบรม
model_cfgs = [
    {
        "name": "efficientnet_b3",
        "img_size": 300,
        "batch_size": 16,
        "head_epochs": 1,
        "full_epochs": 4,
        "head_lr": 1e-3,
        "full_lr": 8e-5,
        "weight_decay": 1e-4,
    },
]

# ฝึกอบรมโมเดล
model_results = []
for cfg in model_cfgs:
    result = train_model(
        cfg,
        train_paths_split,
        train_labels_split,
        val_paths_split,
        val_labels_split,
        device
    )
    model_results.append(result)
    if device.type == "cuda":
        torch.cuda.empty_cache()

log("\nเสร็จสิ้นการฝึกอบรมทั้งหมด!")

## 12. เอนเซมเบิล Blending (Ensemble)

รวมการทำนายของหลายโมเดลด้วยน้ำหนักตามความแม่นยำ

In [ ]:
# สร้างเมทริกซ์ความน่าจะเป็นจากการตรวจสอบ
val_matrix = np.stack([mr["val_probs"] for mr in model_results], axis=1)

# คำนวณน้ำหนักตามความแม่นยำ
raw_weights = np.array(
    [max(1e-6, mr["val_acc"] - 0.5) for mr in model_results],
    dtype=np.float64
)
blend_weights = raw_weights / raw_weights.sum()

# ทำนายแบบเอนเซมเบิล
ensemble_val_probs = val_matrix @ blend_weights
ensemble_acc, ensemble_thr = find_best_threshold(
    val_labels_split,
    ensemble_val_probs
)

log(f"\n[เอนเซมเบิล] ผลลัพธ์การตรวจสอบ:")
log(f"  ความแม่นยำ: {ensemble_acc:.4f}")
log(f"  Threshold: {ensemble_thr:.4f}")
log(f"\nน้ำหนักของโมเดล:")
for i, (mr, w) in enumerate(zip(model_results, blend_weights)):
    log(f"  {mr['name']}: {w:.4f} (val_acc={mr['val_acc']:.4f})")

## 13. การทำนายบนชุดทดสอบ (Test Predictions)

ทำนายบนชุดทดสอบและสร้างไฟล์ submission

In [ ]:
log("\n[ทดสอบ] ทำนายบนชุดทดสอบด้วย TTA...")

test_prob_list = []
for i, mr in enumerate(model_results):
    log(f"  {mr['name']} ({i+1}/{len(model_results)})...")
    probs = predict_with_tta(
        mr["model"],
        test_paths,
        mr["tta_tfms"],
        mr["batch_size"],
        device
    )
    test_prob_list.append(probs)

# เอนเซมเบิล
test_matrix = np.stack(test_prob_list, axis=1)
ensemble_test_probs = test_matrix @ blend_weights
test_preds = (ensemble_test_probs > ensemble_thr).astype(int)

log(f"\nการกระจายของการทำนายทดสอบ:")
unique, counts = np.unique(test_preds, return_counts=True)
for cls, count in zip(unique, counts):
    log(f"  คลาส {cls}: {count} ({count/len(test_preds)*100:.2f}%)")

In [ ]:
# สร้างไฟล์ submission
sample_df["answer"] = test_preds
sample_df.to_csv("submission.csv", index=False)

log(f"\nบันทึกการส่ง: submission.csv")
log(sample_df.head())

## 14. ผลลัพธ์และการวิเคราะห์ (Results Analysis)

สรุปผลลัพธ์และการวิเคราะห์

In [ ]:
# สรุปผลลัพธ์
results_summary = pd.DataFrame([
    {
        "โมเดล": mr["name"],
        "ความแม่นยำ": f"{mr['val_acc']:.4f}",
        "Threshold": f"{mr['val_thr']:.4f}",
        "น้ำหนักเอนเซมเบิล": f"{w:.4f}",
        "Epoch ที่ดี": mr["best_epoch"],
    }
    for mr, w in zip(model_results, blend_weights)
])

print("\n" + "="*80)
print("สรุปผลลัพธ์")
print("="*80)
print(results_summary.to_string(index=False))
print(f"\nเอนเซมเบิล (ชุดตรวจสอบ): {ensemble_acc:.4f}")
print(f"Threshold ของเอนเซมเบิล: {ensemble_thr:.4f}")

In [ ]:
# วาดการเปรียบเทียบความแม่นยำ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ความแม่นยำของโมเดล
model_names = [mr["name"] for mr in model_results]
model_accs = [mr["val_acc"] for mr in model_results]
colors = plt.cm.viridis(np.linspace(0, 1, len(model_names)))

axes[0].bar(model_names, model_accs, color=colors, alpha=0.8, edgecolor='black')
axes[0].axhline(ensemble_acc, color='red', linestyle='--', linewidth=2, label=f'เอนเซมเบิล: {ensemble_acc:.4f}')
axes[0].set_ylabel('ความแม่นยำ (Accuracy)')
axes[0].set_title('ความแม่นยำของโมเดล (ชุดตรวจสอบ)')
axes[0].set_ylim([min(model_accs + [ensemble_acc]) - 0.02, 1.0])
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# น้ำหนักของเอนเซมเบิล
axes[1].bar(model_names, blend_weights, color=colors, alpha=0.8, edgecolor='black')
axes[1].set_ylabel('น้ำหนัก (Weight)')
axes[1].set_title('น้ำหนักของเอนเซมเบิล')
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 15. ดาวน์โหลด Submission (Download Submission)

ดาวน์โหลดไฟล์ submission.csv

In [ ]:
# ดาวน์โหลดไฟล์ submission
from google.colab import files

files.download('submission.csv')
log("\nโหลดไฟล์ submission.csv เสร็จสิ้น!")